In [0]:
# Upgrade Databricks SDK to the latest version and restart Python to see updated packages
%pip install --upgrade databricks-sdk==0.49.0

In [0]:
%restart_python

In [0]:
from databricks.sdk.service.jobs import JobSettings as Job

In [0]:

job_investor_relation_raw_to_curated = Job.from_dict(
    {
        "name": "job_investor_relation_raw_to_curated",
        "tasks": [
            {
                "task_key": "Raw_Load",
                "notebook_task": {
                    "notebook_path": "scripts/orchestration/ir_orchestration_source_to_raw",
                    "source": "GIT",
                },
            },
            {
                "task_key": "Enrich_Investment_Load",
                "depends_on": [
                    {
                        "task_key": "Raw_Load",
                    },
                ],
                "notebook_task": {
                    "notebook_path": "scripts/orchestration/ir_orchestration_raw_to_enrich",
                    "base_parameters": {
                        "table_name": "investment",
                    },
                    "source": "GIT",
                },
                "min_retry_interval_millis": 900000,
                "disable_auto_optimization": True,
            },
            {
                "task_key": "Enrich_Investor_Load",
                "depends_on": [
                    {
                        "task_key": "Raw_Load",
                    },
                ],
                "notebook_task": {
                    "notebook_path": "scripts/orchestration/ir_orchestration_raw_to_enrich",
                    "base_parameters": {
                        "table_name": "investor",
                    },
                    "source": "GIT",
                },
                "min_retry_interval_millis": 900000,
                "disable_auto_optimization": True,
            },
            {
                "task_key": "Enrich_Transaction_Load",
                "depends_on": [
                    {
                        "task_key": "Raw_Load",
                    },
                ],
                "notebook_task": {
                    "notebook_path": "scripts/orchestration/ir_orchestration_raw_to_enrich",
                    "base_parameters": {
                        "table_name": "transaction",
                    },
                    "source": "GIT",
                },
                "min_retry_interval_millis": 900000,
                "disable_auto_optimization": True,
            },
            {
                "task_key": "Curated_Load",
                "depends_on": [
                    {
                        "task_key": "Enrich_Transaction_Load",
                    },
                    {
                        "task_key": "Enrich_Investor_Load",
                    },
                    {
                        "task_key": "Enrich_Investment_Load",
                    },
                ],
                "notebook_task": {
                    "notebook_path": "scripts/orchestration/ir_orchestration_enrich_to_curated",
                    "base_parameters": {
                        "target_table_name": "all",
                    },
                    "source": "GIT",
                },
            },
        ],
        "git_source": {
            "git_url": "https://github.com/anoopdk/medallion_project.git",
            "git_provider": "gitHub",
            "git_branch": "main",
        },
        "queue": {
            "enabled": True,
        },
        "performance_target": "STANDARD",
    }
)




In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
w.jobs.reset(new_settings=job_investor_relation_raw_to_curated, job_id=441984073143903)
# or create a new job using: w.jobs.create(**job_investor_relation_raw_to_curated.as_shallow_dict())